# MiBIG Dataset Preprocessing for BGC Classification

This notebook preprocesses the MiBIG dataset for BGC classification tasks using both 
ESM embeddings and BiGCARP embeddings. It extracts domain sequences from BGCs, converts 
them to appropriate representations, and prepares the data for downstream multilabel 
classification experiments comparing different embedding approaches.

## Overview

The preprocessing pipeline includes:
- Loading and parsing MiBIG dataset
- Converting BGCs to domain sequences  
- Generating embeddings using different approaches
- Preparing data for classification experiments


**step1: Load the dataset and convert to sequence of domain names** 

In [ ]:
# Ensure we're working from repository root for consistent paths
import os
from pathlib import Path

current_dir = Path.cwd()
print(f"Current working directory: {current_dir}")

# Navigate to repo root if we're in a subdirectory
if current_dir.name == "classification":
    os.chdir("../..")  # Go to repo root from notebook_experiments/classification/
    print("Changed to repo root from classification directory")
elif current_dir.name == "notebook_experiments":
    os.chdir("..")  # Go to repo root from notebook_experiments/
    print("Changed to repo root from notebook_experiments directory")
elif "notebook_experiments" in str(current_dir):
    # Find repo root by going up until we don't see notebook_experiments in path
    while "notebook_experiments" in str(Path.cwd()):
        os.chdir("..")
    print("Changed to repo root")

print(f"Working directory is now: {Path.cwd()}")
print(f"Repository structure check - 'data' exists: {Path('data').exists()}")
print(f"Repository structure check - 'artifacts' exists: {Path('artifacts').exists()}")


Current working directory: /lus/lfs1aip2/home/u5bb/han00.u5bb/workspace/cgrep
Working directory is now: /lus/lfs1aip2/home/u5bb/han00.u5bb/workspace/cgrep
Repository structure check - 'data' exists: True
Repository structure check - 'artifacts' exists: True


In [2]:
# 1. to curate the dataset, retrieve the sequences and their corresponding classes from the MiBiG database
import pandas as pd
import numpy as np
from collections import Counter

# Path to the dataset
data_path = 'data/raw/MiBIG_1406_dataset.txt'

# Read the dataset and parse into a DataFrame
def load_mibig_data(file_path):
    bgc_ids = []
    product_classes = []
    pfam_sequences = []
    
    with open(file_path, 'r') as f:
        for line in f:
            # Skip lines that might be comments or headers
            if line.startswith('//') or not line.strip():
                continue
            
            # Split the line into BGC ID, product class, and Pfam domains
            parts = line.strip().split(',')
            if len(parts) >= 3:
                bgc_id = parts[0]
                product_class = parts[1]
                pfams = parts[2].split(';')
                
                bgc_ids.append(bgc_id)
                product_classes.append(product_class)
                pfam_sequences.append(pfams)
    
    # Create a DataFrame
    data = pd.DataFrame({
        'bgc_id': bgc_ids,
        'product_class': product_classes,
        'pfam_sequence': pfam_sequences
    })
    
    return data

# Load the dataset
mibig_data = load_mibig_data(data_path)
# Display the first few rows of the dataset
mibig_data


,bgc_id,product_class,pfam_sequence
0,BGC0000001.1,Polyketide,"[PF02353, PF01135, PF01269, PF13489, PF01596, ..."
1,BGC0000002.1,Polyketide,"[PF00749, PF00201, PF04101, PF13579, PF03033, ..."
2,BGC0000003.1,Polyketide,"[PF00755, PF08659, PF00107, PF13489, PF10294, ..."
3,BGC0000004.1,Polyketide,"[PF07690, PF06609, PF00083, PF00975, PF00550, ..."
4,BGC0000005.1,Polyketide,"[PF00135, PF10340, PF07859, PF12146, PF00975]"
...,...,...,...
1539,BGC0001428.1,NRP;Polyketide,"[PF00150, PF13476, PF00005, PF13555, PF13304, ..."
1540,BGC0001429.1,NRP;Polyketide,"[PF00582, PF00999, PF06826, PF17186, PF07143, ..."
1541,BGC0001430.1,NRP;Polyketide,"[PF00440, PF12146, PF00561, PF12697, PF00756, ..."
1542,BGC0001431.1,NRP;Polyketide,"[PF00582, PF00999, PF06826, PF17186, PF07143, ..."


In [3]:
'''now we have the dataframe mibig_data 
the current pfam sequences are in the format of a list of Pfam IDs
we need to convert them to the domain name format for BiGCARP'''

import json

# Load the domain to Pfam mapping file
mapping_path = 'data/processed/vocabularies/domain_to_pfam_mapping.json'
with open(mapping_path, 'r') as f:
    domain_to_pfam = json.load(f)

# Create reverse mapping: Pfam ID -> domain name
pfam_to_domain = {}
for domain, pfam_id in domain_to_pfam.items():
    # Remove version numbers from Pfam IDs
    pfam_base = pfam_id.split('.')[0] if '.' in pfam_id else pfam_id
    pfam_to_domain[pfam_base] = domain

print(f"Loaded {len(pfam_to_domain)} Pfam ID to domain name mappings")

# Function to convert a list of Pfam IDs to domain names
def convert_pfam_ids_to_domains(pfam_ids):
    domain_names = []
    for pfam_id in pfam_ids:
        # Try to match full ID first, then without version
        if pfam_id in pfam_to_domain:
            domain_names.append(pfam_to_domain[pfam_id])
        else:
            # Try without version number
            pfam_base = pfam_id.split('.')[0] if '.' in pfam_id else pfam_id
            if pfam_base in pfam_to_domain:
                domain_names.append(pfam_to_domain[pfam_base])
            else:
                # If no mapping is found, keep the original ID
                domain_names.append("UNK")
                # Print warning for debugging
                if len(domain_names) < 5:  # Limit to avoid too many messages
                    print(f"Warning: No domain name found for Pfam ID {pfam_id}") # this is probably a problem with the pfam database version. Some of obsolete Pfam IDs are still in the dataset
    
    return domain_names

# Apply the conversion to each sequence in the dataset
mibig_data['domain_sequence'] = mibig_data['pfam_sequence'].apply(convert_pfam_ids_to_domains)

# Check how many Pfams were successfully mapped
mapped_count = sum(1 for seq in mibig_data['domain_sequence'] for domain in seq if domain != "UNK")
total_count = sum(len(seq) for seq in mibig_data['domain_sequence'])
print(f"Successfully mapped {mapped_count} out of {total_count} Pfam IDs ({mapped_count/total_count*100:.2f}%)")

# Preview a few examples of the conversion
print("\nExample conversions:")
for i in range(min(3, len(mibig_data))):
    print(f"BGC ID: {mibig_data.iloc[i]['bgc_id']}")
    print(f"First 5 Pfam IDs: {mibig_data.iloc[i]['pfam_sequence'][:5]}")
    print(f"Converted to domains: {mibig_data.iloc[i]['domain_sequence'][:5]}")
    print("-----")

# Save the updated dataset with domain sequences
output_dir = "artifacts/classification/mibig1/"
filename = "mibig1_preprocessed_pfam_seq.pkl"
save_path = output_dir + filename

mibig_data.to_pickle(save_path)
print(f"MiBiG dataset saved to: {save_path}")



Loaded 24076 Pfam ID to domain name mappings
Successfully mapped 74631 out of 74824 Pfam IDs (99.74%)

Example conversions:
BGC ID: BGC0000001.1
First 5 Pfam IDs: ['PF02353', 'PF01135', 'PF01269', 'PF13489', 'PF01596']
Converted to domains: ['CMAS', 'PCMT', 'Fibrillarin', 'Methyltransf_23', 'Methyltransf_3']
-----
BGC ID: BGC0000002.1
First 5 Pfam IDs: ['PF00749', 'PF00201', 'PF04101', 'PF13579', 'PF03033']
Converted to domains: ['tRNA-synt_1c', 'UDPGT', 'Glyco_tran_28_C', 'Glyco_trans_4_4', 'Glyco_transf_28']
-----
BGC ID: BGC0000003.1
First 5 Pfam IDs: ['PF00755', 'PF08659', 'PF00107', 'PF13489', 'PF10294']
Converted to domains: ['Carn_acyltransf', 'KR', 'ADH_zinc_N', 'Methyltransf_23', 'Methyltransf_16']
-----
MiBiG dataset saved to: artifacts/classification/mibig1/mibig1_preprocessed_pfam_seq.pkl


**step2: convert the domain sequence to sequences of embeddings using esm and bc**

In [4]:
import torch
import json
from torch.utils.data import DataLoader, Dataset
from sequence_models.convolutional import ByteNetLM
from tqdm import tqdm
from cgrep import utils
import pandas as pd
import os
from pathlib import Path


/home/u5bb/han00.u5bb/miniforge3/envs/cgrep/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. BIGCARP embeddings

In [5]:
import os
import json
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from tqdm import tqdm
from typing import Sequence
from sequence_models.convolutional import ByteNetLM
from cgrep import utils


# ------------------------------------------------------------------
# MAIN FUNCTION
# ------------------------------------------------------------------
def extract_mibig_embeddings(
    ckpt_path: str,
    bigcarp_vocab_path: str,
    mibig_data_path: str,
    output_path: str,
    layer_indices: Sequence[str] = ["last"],   # <- use typing.Sequence
    frozen: bool = False,
) -> None:
    """
    Extract layer embeddings from a single BIGCARP checkpoint
    and save them alongside the MiBiG dataframe.

    Parameters
    ----------
    ckpt_path : str
        Path to the specific BIGCARP checkpoint file.
    bigcarp_vocab_path : str
        JSON file with the BIGCARP vocabulary (must include "specials" and "domains").
    mibig_data_path : str
        Pickled pandas DataFrame containing MiBiG sequences under 'domain_sequence'.
    output_path : str
        Where to write the pickled DataFrame with embeddings.
    layer_indices : sequence[str], default ("last",)
        Which transformer layers to extract (passed to utils.extract_layer_embeddings).
    frozen : bool, default False
        Whether to build a ByteNetLM with frozen embeddings.
    """

    # ------------------------------------------------------------------
    # 1. Load vocab
    # ------------------------------------------------------------------
    with open(bigcarp_vocab_path, "r") as f:
        vocab_info = json.load(f)
    specials  = vocab_info["specials"]
    domains   = vocab_info["domains"]
    padding_idx = specials["-"]
    mask_idx    = specials["#"]
    n_tokens = vocab_info["size"]

    # ------------------------------------------------------------------
    # 2. Load & tokenize MiBiG
    # ------------------------------------------------------------------
    mibig_data = pd.read_pickle(mibig_data_path)

    def tokenize_sequences(domain_sequences, domain_to_token):
        tokenized_sequences = []
        for seq in tqdm(domain_sequences, desc="Tokenizing sequences"):
            tokenized_sequences.append(
                [domain_to_token.get(domain, domain_to_token["UNK"]) for domain in seq]
            )
        return tokenized_sequences

    if "tokenized_sequence" not in mibig_data.columns:
        mibig_data["tokenized_sequence"] = tokenize_sequences(
            mibig_data["domain_sequence"], domains
        )

    # ------------------------------------------------------------------
    # 3. Prepare dataset/dataloader
    # ------------------------------------------------------------------
    class PfamDataset(Dataset):
        def __init__(self, tokenized_sequences):
            self.tokenized_sequences = tokenized_sequences

        def __len__(self):
            return len(self.tokenized_sequences)

        def __getitem__(self, idx):
            return torch.tensor(self.tokenized_sequences[idx], dtype=torch.long)

    dataset = PfamDataset(mibig_data["tokenized_sequence"].tolist())
    dataloader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        collate_fn=lambda b: utils.mlm_collate_fn_extraction(
            b, mask_idx, padding_idx, mask_frac=0
        ),
    )

    # ------------------------------------------------------------------
    # 4. Device
    # ------------------------------------------------------------------
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Ensure output directory exists
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------------
    # 5. Process single checkpoint
    # ------------------------------------------------------------------
    # Build model (bc mode)
    model_kwargs = dict(
        n_tokens=n_tokens,
        d_embedding=1280,
        d_model=256,
        n_layers=32,
        kernel_size=3,
        r=128,
        slim=True,
        padding_idx=mask_idx,
        causal=False,
        final_ln=True,
        activation="gelu",
    )
    print("built model in bc mode")
    if frozen:
        model_kwargs["n_frozen_embs"] = len(domains) - 1

    model = ByteNetLM(**model_kwargs)
    
    # Load checkpoint
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
        
    ckpt = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval().to(device)

    # Extract embeddings
    all_embeddings = []
    for batch_tuple in tqdm(dataloader, desc="Extracting embeddings"):
        batch, mask = batch_tuple
        batch = batch.to(device)
        mask = mask.to(device)
        # Create attention mask: exclude only real padding (bc mode)
        input_mask = (batch != padding_idx).float().unsqueeze(-1)

        emb = utils.extract_layer_embeddings(
            model, batch, input_mask=input_mask, layer_indices=layer_indices
        )
        all_embeddings.extend(emb.detach().cpu().numpy())

    # Attach & save
    mibig_data["embeddings"] = all_embeddings
    mibig_data.to_pickle(output_path)
    
    # GPU housekeeping
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"Embeddings saved to: {output_path}")

In [31]:
# the esm-pretrained initialization embs last layer
extract_mibig_embeddings(
    ckpt_path="artifacts/bigcarp/bigcarp_models/run_esm_init/checkpoint_best.tar",
    bigcarp_vocab_path="data/processed/vocabularies/pfam_vocab_present.json",
    mibig_data_path="artifacts/classification/mibig1/mibig1_preprocessed_pfam_seq.pkl",
    output_path="artifacts/classification/mibig1/mibig1_esm_last.pkl",
    layer_indices=["last"],
    frozen=False,
)

Tokenizing sequences: 100%|██████████| 1544/1544 [00:00<00:00, 200266.12it/s]

built model in bc mode



Extracting embeddings: 100%|██████████| 1544/1544 [00:30<00:00, 50.16it/s]


Embeddings saved to: artifacts/classification/mibig1/mibig1_esm_last.pkl


In [32]:
# the esm-pretrained initialization embs embedder
extract_mibig_embeddings(
    ckpt_path="artifacts/bigcarp/bigcarp_models/run_esm_init/checkpoint_best.tar",
    bigcarp_vocab_path="data/processed/vocabularies/pfam_vocab_present.json",
    mibig_data_path="artifacts/classification/mibig1/mibig1_preprocessed_pfam_seq.pkl",
    output_path="artifacts/classification/mibig1/mibig1_esm_embedder.pkl",
    layer_indices=["embedder"],
    frozen=False,
)

Tokenizing sequences: 100%|██████████| 1544/1544 [00:00<00:00, 194369.57it/s]


built model in bc mode


Extracting embeddings: 100%|██████████| 1544/1544 [00:00<00:00, 4206.71it/s]


Embeddings saved to: artifacts/classification/mibig1/mibig1_esm_embedder.pkl


In [33]:
# the random initialization embs last layer
extract_mibig_embeddings(
    ckpt_path="artifacts/bigcarp/bigcarp_models/run_random_init/checkpoint_best.tar",
    bigcarp_vocab_path="data/processed/vocabularies/pfam_vocab_present.json",
    mibig_data_path="artifacts/classification/mibig1/mibig1_preprocessed_pfam_seq.pkl",
    output_path="artifacts/classification/mibig1/mibig1_random_last.pkl",
    layer_indices=["last"],
    frozen=False,
)

Tokenizing sequences: 100%|██████████| 1544/1544 [00:00<00:00, 173456.69it/s]


built model in bc mode


Extracting embeddings:   1%|          | 11/1544 [00:00<00:42, 36.00it/s]

Extracting embeddings: 100%|██████████| 1544/1544 [00:30<00:00, 50.86it/s]


Embeddings saved to: artifacts/classification/mibig1/mibig1_random_last.pkl


In [34]:
# the random initialization embs embedder
extract_mibig_embeddings(
    ckpt_path="artifacts/bigcarp/bigcarp_models/run_random_init/checkpoint_best.tar",
    bigcarp_vocab_path="data/processed/vocabularies/pfam_vocab_present.json",
    mibig_data_path="artifacts/classification/mibig1/mibig1_preprocessed_pfam_seq.pkl",
    output_path="artifacts/classification/mibig1/mibig1_random_embedder.pkl",
    layer_indices=["embedder"],
    frozen=False,
)

Tokenizing sequences: 100%|██████████| 1544/1544 [00:00<00:00, 205646.23it/s]


built model in bc mode


Extracting embeddings: 100%|██████████| 1544/1544 [00:00<00:00, 4147.71it/s]


Embeddings saved to: artifacts/classification/mibig1/mibig1_random_embedder.pkl


## try using fixed checkpoints.

In [13]:
# the esm-pretrained initialization embs last layer after 99 epochs
extract_mibig_embeddings(
    ckpt_path="artifacts/bigcarp/bigcarp_models/run_esm_init/checkpoint_epoch99.tar",
    bigcarp_vocab_path="data/processed/vocabularies/pfam_vocab_present.json",
    mibig_data_path="artifacts/classification/mibig1/mibig1_preprocessed_pfam_seq.pkl",
    output_path="artifacts/classification/mibig1/mibig1_esm_last_99.pkl",
    layer_indices=["last"],
    frozen=False,
)

Tokenizing sequences: 100%|██████████| 1544/1544 [00:00<00:00, 200557.61it/s]

built model in bc mode


Extracting embeddings: 100%|██████████| 1544/1544 [00:33<00:00, 46.45it/s]


Embeddings saved to: artifacts/classification/mibig1/mibig1_esm_last_99.pkl


In [14]:
# the esm-pretrained initialization embs embedder after 99 epochs
extract_mibig_embeddings(
    ckpt_path="artifacts/bigcarp/bigcarp_models/run_esm_init/checkpoint_epoch99.tar",
    bigcarp_vocab_path="data/processed/vocabularies/pfam_vocab_present.json",
    mibig_data_path="artifacts/classification/mibig1/mibig1_preprocessed_pfam_seq.pkl",
    output_path="artifacts/classification/mibig1/mibig1_esm_embedder_99.pkl",
    layer_indices=["embedder"],
    frozen=False,
)


Tokenizing sequences: 100%|██████████| 1544/1544 [00:00<00:00, 205888.13it/s]


built model in bc mode


Extracting embeddings: 100%|██████████| 1544/1544 [00:00<00:00, 3413.13it/s]


Embeddings saved to: artifacts/classification/mibig1/mibig1_esm_embedder_99.pkl


In [15]:

# the random initialization embs last layer after 99 epochs
extract_mibig_embeddings(
    ckpt_path="artifacts/bigcarp/bigcarp_models/run_random_init/checkpoint_epoch99.tar",
    bigcarp_vocab_path="data/processed/vocabularies/pfam_vocab_present.json",
    mibig_data_path="artifacts/classification/mibig1/mibig1_preprocessed_pfam_seq.pkl",
    output_path="artifacts/classification/mibig1/mibig1_random_last_99.pkl",
    layer_indices=["last"],
    frozen=False,
)
# the random initialization embs embedder after 99 epochs
extract_mibig_embeddings(
    ckpt_path="artifacts/bigcarp/bigcarp_models/run_random_init/checkpoint_epoch99.tar",
    bigcarp_vocab_path="data/processed/vocabularies/pfam_vocab_present.json",
    mibig_data_path="artifacts/classification/mibig1/mibig1_preprocessed_pfam_seq.pkl",
    output_path="artifacts/classification/mibig1/mibig1_random_embedder_99.pkl",
    layer_indices=["embedder"],
    frozen=False,
)




Tokenizing sequences: 100%|██████████| 1544/1544 [00:00<00:00, 172087.73it/s]


built model in bc mode


Extracting embeddings: 100%|██████████| 1544/1544 [00:32<00:00, 47.38it/s]


Embeddings saved to: artifacts/classification/mibig1/mibig1_random_last_99.pkl


Tokenizing sequences: 100%|██████████| 1544/1544 [00:00<00:00, 203981.52it/s]


built model in bc mode


Extracting embeddings: 100%|██████████| 1544/1544 [00:00<00:00, 3438.49it/s]


Embeddings saved to: artifacts/classification/mibig1/mibig1_random_embedder_99.pkl


#### 2) ESM Embeddings

In [35]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import json
import torch
# load the embeddings from disk
mibig_data = pd.read_pickle("artifacts/classification/mibig1/mibig1_preprocessed_pfam_seq.pkl")

In [36]:
# load the existing embeddings of esm-based pfam domains
emb_matrix = torch.load("artifacts/bigcarp/esm_embeddings/esm1b_pfam_embs.pt")

# load the vocab of pfam domains which encodes the domains' positions in the embedding matrix
vocab = json.load(open("data/processed/vocabularies/pfam_vocab.json"))
domains = vocab["domains"]
# remove the UNK domain
domains = {k: v for k, v in domains.items() if k != "UNK"}
# create a mapping from domain to index
domain_to_idx = {domain: idx for idx, domain in enumerate(domains.keys())}


def get_domain_embeddings_esm(domain_sequence, domain_to_idx, emb_matrix):
    """
    Given a sequence of domain names, return the corresponding embeddings from the ESM's embeddings matrix.
    """
    # Get the indices for the domains in the sequence
    indices = [domain_to_idx[domain] for domain in domain_sequence if domain in domain_to_idx]
    # Get the corresponding embeddings
    embeddings = emb_matrix[indices]
    return embeddings

# Apply the function to the dataset
mibig_data['esm_embeddings'] = mibig_data['domain_sequence'].apply(
    lambda seq: get_domain_embeddings_esm(seq, domain_to_idx, emb_matrix)
)


In [37]:
# Sanity check: empty embeddings ()
# Check for empty embeddings
empty_embeddings = mibig_data['esm_embeddings'].apply(lambda emb: emb.shape[0] == 0)

# Print the number of entries with empty embeddings
print(f"Number of entries with empty embeddings: {empty_embeddings.sum()}")

# Optionally, display the rows with empty embeddings
if empty_embeddings.any():
    print("Entries with empty embeddings:")
    print(mibig_data[empty_embeddings])

Number of entries with empty embeddings: 1
Entries with empty embeddings:
           bgc_id product_class pfam_sequence domain_sequence esm_embeddings
380  BGC0000361.1           NRP     [PF08982]          [AtaL]             []


In [38]:
# save the embeddings to disk
mibig_data.to_pickle("artifacts/classification/mibig1/mibig_esm_embeddings.pkl")

In [39]:
# Check the head of the mibig_data DataFrame
mibig_data.head()


,bgc_id,product_class,pfam_sequence,domain_sequence,esm_embeddings
0,BGC0000001.1,Polyketide,"[PF02353, PF01135, PF01269, PF13489, PF01596, ...","[CMAS, PCMT, Fibrillarin, Methyltransf_23, Met...","[[tensor(-0.0313), tensor(0.1117), tensor(-0.2..."
1,BGC0000002.1,Polyketide,"[PF00749, PF00201, PF04101, PF13579, PF03033, ...","[tRNA-synt_1c, UDPGT, Glyco_tran_28_C, Glyco_t...","[[tensor(0.0172), tensor(-0.0121), tensor(-0.2..."
2,BGC0000003.1,Polyketide,"[PF00755, PF08659, PF00107, PF13489, PF10294, ...","[Carn_acyltransf, KR, ADH_zinc_N, Methyltransf...","[[tensor(-0.0030), tensor(0.2396), tensor(0.21..."
3,BGC0000004.1,Polyketide,"[PF07690, PF06609, PF00083, PF00975, PF00550, ...","[MFS_1, TRI12, Sugar_tr, Thioesterase, PP-bind...","[[tensor(-0.0105), tensor(0.1228), tensor(-0.0..."
4,BGC0000005.1,Polyketide,"[PF00135, PF10340, PF07859, PF12146, PF00975]","[COesterase, Say1_Mug180, Abhydrolase_3, Hydro...","[[tensor(-0.0700), tensor(0.1653), tensor(-0.0..."


#### 3) pfam2vec Embeddings

In [40]:
import pandas as pd

mibig_data = pd.read_pickle("artifacts/classification/mibig1/mibig_esm_embeddings.pkl")
# Create mibig_data_pfam by copying mibig_data
mibig_data_pfam = mibig_data.copy()

pfam_path = 'data/processed/bgc_product_classification/pfam2vec.csv'
pfam_df = pd.read_csv(pfam_path)

In [41]:
import numpy as np
# Function to convert PFAM sequence to sequence of embeddings
def get_pfam_sequence_embeddings_and_missing(pfam_sequence, pfam_df):
    """
    Convert a sequence of PFAM domains to a sequence of their corresponding embeddings,
    and report any domains not found in pfam_df.
    
    Args:
        pfam_sequence: List or string of PFAM domains
        pfam_df: DataFrame containing PFAM domain embeddings. 
                 Expected to have 'pfam_id' as one column and embedding vectors in others.
        
    Returns:
        Tuple: (List of embedding vectors, List of missing PFAM IDs for this sequence)
    """
    # Parse the pfam_sequence if it's a string
    if isinstance(pfam_sequence, list):
        domains = pfam_sequence
    elif isinstance(pfam_sequence, str):
        domains = pfam_sequence.split(';') if ';' in pfam_sequence else \
                 pfam_sequence.split(',') if ',' in pfam_sequence else [pfam_sequence]
        domains = [d.strip() for d in domains if d.strip()]
    else:
        return [], [] # Return empty lists if input is not list or string
    
    embeddings = []
    missing_domains_in_sequence = []
    for domain in domains:
        # Ensure 'pfam_id' column exists in pfam_df
        if 'pfam_id' not in pfam_df.columns:
            # This is a critical error in pfam_df structure, handle it
            print("Error: 'pfam_id' column not found in pfam_df. Cannot proceed.")
            # Depending on desired behavior, you might raise an error or return.
            # For this example, we'll assume this sequence can't be processed further.
            return [], domains # Return all domains as missing for this sequence if pfam_id col is absent
            
        domain_match = pfam_df[pfam_df['pfam_id'] == domain]
        if not domain_match.empty:
            # Get the embedding vector (all columns except the pfam_id)
            # We assume pfam_id is the first column. If not, this logic needs adjustment.
            # A safer way is to select by column names if they are consistent,
            # or ensure pfam_id is an index for easier lookup.
            embedding = domain_match.iloc[0, pfam_df.columns != 'pfam_id'].values.astype(np.float32)
            embeddings.append(embedding)
        else:
            # If domain not found, print warning and add to missing list
            print(f"Warning: Domain '{domain}' not found in pfam_df embeddings.")
            missing_domains_in_sequence.append(domain)
    
    return embeddings, missing_domains_in_sequence


In [42]:
# Apply the function to create a new column with embeddings
results_tuples = mibig_data_pfam['pfam_sequence'].apply(
    lambda x: get_pfam_sequence_embeddings_and_missing(x, pfam_df)
)

# Separate the tuples into two new columns or Series
mibig_data_pfam['pfam2vec_seq'] = results_tuples.str[0]
list_of_missing_pfams_per_bgc = results_tuples.str[1]

# --- Summarize unique missing PFAM IDs ---
overall_unique_missing_pfams = set()
for missing_list in list_of_missing_pfams_per_bgc:
    if missing_list: # Ensure the list is not None or empty
        overall_unique_missing_pfams.update(missing_list)

if overall_unique_missing_pfams:
    print("\n--------------------------------------------------------------------")
    print("--- Summary of Unique PFAM IDs Not Found in Embedding DataFrame ---")
    print(f"A total of {len(overall_unique_missing_pfams)} unique PFAM IDs from your sequences were not found in pfam_df:")
    for pfam_id in sorted(list(overall_unique_missing_pfams)):
        print(f"  - {pfam_id}")
    print("For BGCs containing these PFAM IDs, the missing domains were skipped during embedding generation.")
    print("--------------------------------------------------------------------")
else:
    print("\n--------------------------------------------------------------------")
    print("All PFAM IDs from your sequences were successfully found in the pfam_df embeddings.")
    print("--------------------------------------------------------------------")



--------------------------------------------------------------------
--- Summary of Unique PFAM IDs Not Found in Embedding DataFrame ---
A total of 64 unique PFAM IDs from your sequences were not found in pfam_df:
  - PF00098
  - PF00172
  - PF00241
  - PF00262
  - PF00399
  - PF00789
  - PF00974
  - PF01328
  - PF02072
  - PF02184
  - PF02197
  - PF02721
  - PF02953
  - PF03723
  - PF03939
  - PF04027
  - PF04082
  - PF04419
  - PF04774
  - PF05032
  - PF05254
  - PF05363
  - PF05383
  - PF05699
  - PF05730
  - PF06985
  - PF07428
  - PF07524
  - PF07712
  - PF07915
  - PF08059
  - PF08130
  - PF08195
  - PF08390
  - PF08699
  - PF09005
  - PF09368
  - PF10200
  - PF10270
  - PF10406
  - PF10430
  - PF10585
  - PF10587
  - PF12013
  - PF12108
  - PF12593
  - PF12786
  - PF12907
  - PF13015
  - PF13323
  - PF13917
  - PF13947
  - PF13966
  - PF14380
  - PF14746
  - PF15519
  - PF15725
  - PF16010
  - PF16086
  - PF16275
  - PF16900
  - PF17046
  - PF17064
  - PF17584
For BGCs containi

In [43]:
import pickle
with open('artifacts/classification/mibig1/mibig_embeddings_p2v.pkl', 'wb') as f:
    pickle.dump(mibig_data_pfam, f)
# Check the column names and basic info of mibig_data

# mibig_data_pfam.to_csv('artifacts/classification/mibig1/mibig_embeddings_p2v.csv', index=False)

mibig_data_pfam.head()

,bgc_id,product_class,pfam_sequence,domain_sequence,esm_embeddings,pfam2vec_seq
0,BGC0000001.1,Polyketide,"[PF02353, PF01135, PF01269, PF13489, PF01596, ...","[CMAS, PCMT, Fibrillarin, Methyltransf_23, Met...","[[tensor(-0.0313), tensor(0.1117), tensor(-0.2...","[[0.09173657, 0.035680633, 0.0135731, -0.12299..."
1,BGC0000002.1,Polyketide,"[PF00749, PF00201, PF04101, PF13579, PF03033, ...","[tRNA-synt_1c, UDPGT, Glyco_tran_28_C, Glyco_t...","[[tensor(0.0172), tensor(-0.0121), tensor(-0.2...","[[0.14181772, -0.007126024, 0.007510652, -0.07..."
2,BGC0000003.1,Polyketide,"[PF00755, PF08659, PF00107, PF13489, PF10294, ...","[Carn_acyltransf, KR, ADH_zinc_N, Methyltransf...","[[tensor(-0.0030), tensor(0.2396), tensor(0.21...","[[-0.018139565, -0.08186026, -0.14477561, -0.0..."
3,BGC0000004.1,Polyketide,"[PF07690, PF06609, PF00083, PF00975, PF00550, ...","[MFS_1, TRI12, Sugar_tr, Thioesterase, PP-bind...","[[tensor(-0.0105), tensor(0.1228), tensor(-0.0...","[[-0.03312874, 0.0647246, -0.062299524, -0.002..."
4,BGC0000005.1,Polyketide,"[PF00135, PF10340, PF07859, PF12146, PF00975]","[COesterase, Say1_Mug180, Abhydrolase_3, Hydro...","[[tensor(-0.0700), tensor(0.1653), tensor(-0.0...","[[0.06837933, 0.071370736, -0.113139234, -0.02..."
